# Alphamind新手入门之二：因子排序和分位数

alpha-mind的data文件夹提供了对于因子数据进行排序和求分位数的工具函数

### 一、因子排序： *rank*
- 从小到大排序，返回序列值。
- 可以进行整体排序，也可以分行业(分组)排序。

In [ ]:
# 假设有10只股票，每只股票有2个因子，构成一个矩阵
import asyncio
import datetime as dt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc


In [ ]:
# 假设有10只股票，每只股票有1个因子
# 假设这10只股票分为两个行业,前5个和后5个分属不同类别
rng = np.random.default_rng(42)
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


### 二、因子分位数: *quantile*
- 根据给定组数*(n_bins)*，按从小达到的顺序进行分组，返回每个因子属于的组别。

In [ ]:
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.data.quantile import quantile
from alphamind.data.rank import percentile, rank


In [ ]:
latest = panel.filter(pc.equal(panel["trade_date"], dates[-1]))
signal = latest["factor"].to_numpy(zero_copy_only=False)
result = pa.table({
    "code": latest["code"],
    "rank": rank(signal).flatten(),
    "percentile": percentile(signal).flatten(),
    "quantile": quantile(signal, n_bins=4),
})
assert result.num_rows == n_assets
result
